# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SehrishEjaz1/Flyrank_ML_Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess
if not os.path.exists("Flyrank_ML_Intern"):
    subprocess.run(["git", "clone", "https://github.com/SehrishEjaz1/Flyrank_ML_Intern.git"])
os.chdir("Flyrank_ML_Intern")

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2: Refresh / Content Opportunity Scoring

Task type: Classification

I am predicting which pages are declining — yes or no.
This makes it a binary classification problem.
Each page either needs review (1) or does not (0).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label

A page is labelled 1 when trend_direction == "down".
This is a proxy label — it comes from a defined rule,
not a directly observed future outcome.
It is a starting point, not a perfect truth.

In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts())
print("Decline rate:", round(df["is_declining_label"].mean()*100,1), "%")

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Decline rate: 54.2 %


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50

Of the top 50 pages the model flags for review,
how many are actually declining?

The starter baseline gets about 12 of 50 right (0.240).
A good model should get 30+ of 50 right (0.600+).
Precision@50 matches how the list is actually used —
a reviewer checks the top pages first.

In [3]:
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Baseline: random score
random_scores = np.random.rand(len(df))
baseline = precision_at_k(random_scores, df["is_declining_label"].values)
print(f"Random baseline Precision@50: {baseline:.3f}")
print(f"Target to beat: 0.600+")

Random baseline Precision@50: 0.560
Target to beat: 0.600+


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one page (one row = one webpage)

Each row represents a single content page with its
search and engagement signals from the last 90 days.

In [4]:
features = ["impressions_90d", "avg_position", "ctr",
            "days_since_last_update", "content_age_days",
            "word_count", "is_declining_label"]

sample = df[features].head(5)
print("One row = one page:")
print(sample)

One row = one page:
   impressions_90d  avg_position   ctr  days_since_last_update  \
0             3803          10.6  0.76                      20   
1            15320          20.3  0.05                      25   
2            12581          36.5  0.09                      20   
3            11751           6.2  0.49                      22   
4            19140          44.0  0.13                      14   

   content_age_days  word_count  is_declining_label  
0               187      3221.0                   1  
1               445      2481.0                   1  
2               141      3515.0                   1  
3               463         NaN                   0  
4               263      2803.0                   1  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag pages older than 180 days"
misses too much context. A page can be old but still
performing well — or new but already declining.

ML looks at multiple signals together: age, impressions,
position, CTR, and engagement. The combination of signals
matters more than any single threshold.

A decision tree can find patterns like:
"pages with high impressions AND low CTR AND stale content
are most likely declining" — which no simple if-statement captures.

In [5]:
# Show why single rules miss the pattern
old_pages = df[df["content_age_days"] >= 180]
print("Old pages (180+ days):")
print("Total:", len(old_pages))
print("Declining:", old_pages["is_declining_label"].sum())
print("Decline rate:", round(old_pages["is_declining_label"].mean()*100,1), "%")

print("\nAll pages decline rate:", round(df["is_declining_label"].mean()*100,1), "%")
print("Age alone is not enough signal.")

Old pages (180+ days):
Total: 17986
Declining: 8739
Decline rate: 48.6 %

All pages decline rate: 54.2 %
Age alone is not enough signal.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.